In [1]:
import os
import json
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import ast
import pandas as pd
import string

In [10]:
COLOR_MAPPING = {
    (0xFF, 0xD7, 0x00): 'common room',
    (0xFF, 0xA5, 0x00): 'master room',
    (0xEE, 0xE8, 0xAA): 'living room',
    (0x6B, 0x8E, 0x23): 'balcony',
    (0xAD, 0xD8, 0xE6): 'bathroom',
    (0xF0, 0x80, 0x80): 'kitchen',
    (0xDD, 0xA0, 0xDD): 'storage',
    (0xDA, 0x70, 0xD6): 'dining',
}

In [2]:
# Paths
base_dir = Path("../..")
# json_path = "groups_same_first.json"
csv_path = "groups_same_first_global_results.csv"
img_dir = base_dir / "data" / "floorplan_image"
txt_dir = base_dir / "annotation" / "human_annotated_tags"
out_dir = Path(".") / "examples_output"
out_dir.mkdir(exist_ok=True)

# Load JSON entries
# with open(json_path, 'r') as f:
#     entries = json.load(f)

# Prepare font for subtitles
try:
    font = ImageFont.truetype("arial.ttf", size=16)
except IOError:
    font = ImageFont.load_default()

subtitle_height = 20  # space for ID subtitles

df = pd.read_csv(csv_path)
entries = df['options'].apply(lambda s: ast.literal_eval(s))

for idx, entry in enumerate(entries, start=1):
    group_ids = entry
    images = []

    # Load images for this group
    for img_id in group_ids:
        img_path = img_dir / f"{img_id}.png"
        if img_path.exists():
            images.append((img_id, Image.open(img_path)))
        else:
            raise FileNotFoundError(f"Image file not found: {img_path}")

    # Determine layout
    widths, heights = zip(*(im.size for _, im in images))
    total_width = sum(widths)
    max_height = max(heights)

    # Create canvas
    canvas = Image.new('RGB', (total_width, max_height + subtitle_height), color=(255, 255, 255))
    draw = ImageDraw.Draw(canvas)

    # Paste images and draw subtitles
    x_offset = 0
    for img_id, im in images:
        canvas.paste(im, (x_offset, 0))
        # Compute text size
        if hasattr(font, 'getsize'):
            text_w, text_h = font.getsize(img_id)
        else:
            bbox = draw.textbbox((0,0), img_id, font=font)
            text_w, text_h = bbox[2] - bbox[0], bbox[3] - bbox[1]
        text_x = x_offset + (im.width - text_w) // 2
        text_y = max_height + (subtitle_height - text_h) // 2
        draw.text((text_x, text_y), img_id, fill=(0, 0, 0), font=font)
        x_offset += im.width

    # Save combined image
    img_out_path = out_dir / f"example_{idx}_2.png"
    canvas.save(img_out_path)

    # # Collect and write text descriptions
    # txt_out_path = out_dir / f"example_{idx}.txt"
    # with open(txt_out_path, 'w') as txt_out:
    #     for img_id, _ in images:
    #         tag_file = txt_dir / f"{img_id}.txt"
    #         if tag_file.exists():
    #             with open(tag_file, 'r') as tf:
    #                 desc = tf.read().strip()
    #         else:
    #             raise FileNotFoundError(f"Tag file not found: {tag_file}")
    #         txt_out.write(f"ID: {img_id}, description: {desc}\n\n")

print(f"Generated {len(entries)} example image/text pairs in '{out_dir.resolve()}'")



KeyError: 'options'

In [3]:
import glob

In [36]:
input_folder  = 'examples_same_second'
output_folder = os.path.join(input_folder, 'combined')
os.makedirs(output_folder, exist_ok=True)

# Find every file that ends with '_2.png'
pattern = os.path.join(input_folder, '*_2.png')
for suffix_path in glob.glob(pattern):
    # Derive the base filename by stripping off '_2' before the extension
    folder, suffix_fn = os.path.split(suffix_path)
    stem = suffix_fn[:-6]  # removes the trailing "_2.png"
    base_fn = stem + '.png'
    base_path = os.path.join(folder, base_fn)

    if not os.path.exists(base_path):
        print(f"No base image found for {suffix_fn}, skipping.")
        continue

    # Open both images (they must be same size)
    img_base   = Image.open(base_path)
    img_suffix = Image.open(suffix_path)
    w, h       = img_base.size

    # Stack base on top of suffix
    combined = Image.new('RGB', (w, 2*h))
    combined.paste(img_base,   (0, 0))
    combined.paste(img_suffix, (0, h))

    # Save
    out_fn = f"{stem}_combined.png"
    combined.save(os.path.join(output_folder, out_fn))



No base image found for example_2.png, skipping.


In [6]:
base_dir = Path("../..")
csv_path = "groups_same_second_results.csv"
img_dir = base_dir / "data" / "floorplan_image"
out_dir = Path(".") / "examples_output"
out_dir.mkdir(exist_ok=True)

# Try to load a scalable font at 24px
font_size = 24
for font_name in ("arial.ttf", "DejaVuSans.ttf"):
    try:
        font = ImageFont.truetype(font_name, size=font_size)
        break
    except IOError:
        font = None
if font is None:
    # last resort: default (will ignore size)
    font = ImageFont.load_default()
    print("Warning: falling back to default font; letter size may not change.")

subtitle_height = font_size + 8  # give a bit of padding

# Read groups
df = pd.read_csv(csv_path)
entries = df['options'].apply(lambda s: ast.literal_eval(s))

# Prepare A, B, C… labels
letters = list(string.ascii_uppercase)

for idx, entry in enumerate(entries, start=1):
    images = []

    # Load images for this group
    for img_id in entry:
        img_path = img_dir / f"{img_id}.png"
        if not img_path.exists():
            raise FileNotFoundError(f"Image file not found: {img_path}")
        images.append(Image.open(img_path))

    # Compute canvas size
    widths, heights = zip(*(im.size for im in images))
    total_width = sum(widths)
    max_height = max(heights)

    # Create canvas
    canvas = Image.new("RGB", (total_width, max_height + subtitle_height), "white")
    draw = ImageDraw.Draw(canvas)

    # Paste & subtitle
    x = 0
    for i, im in enumerate(images):
        canvas.paste(im, (x, 0))
        label = letters[i]
        # measure
        if hasattr(font, "getsize"):
            w, h = font.getsize(label)
        else:
            bx0, by0, bx1, by1 = draw.textbbox((0,0), label, font=font)
            w, h = bx1 - bx0, by1 - by0
        tx = x + (im.width - w) // 2
        ty = max_height + (subtitle_height - h) // 2
        draw.text((tx, ty), label, fill="black", font=font)
        x += im.width

    # Save
    canvas.save(out_dir / f"example_{idx}_2.png")

print(f"Generated {len(entries)} example image groups in '{out_dir.resolve()}'")

Generated 100 example image groups in '/home/airlay88/planscape/examples/complex/examples_output'


In [11]:
def add_legend_tight(
    image: Image.Image,
    mapping: dict,
    legend_gap: int = 75,
    font_size: int = 16
) -> Image.Image:
    swatch  = 20
    pad_x   = 5
    pad_y   = 2
    pad_mid = legend_gap

    # try to load a truetype font at the requested size, fallback to default
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", font_size)
    except IOError:
        font = ImageFont.load_default()

    entries = list(mapping.items())

    # measure text widths
    dummy     = Image.new("RGB", (1,1))
    draw0     = ImageDraw.Draw(dummy)
    text_widths = [
        draw0.textbbox((0,0), label, font=font)[2]
        for _, label in entries
    ]

    # compute legend dims
    entry_widths   = [swatch + pad_x + w for w in text_widths]
    total_legend_w = sum(entry_widths) + pad_x * (len(entries) + 1)
    legend_h       = swatch + 2 * pad_y

    # render legend
    legend = Image.new("RGB", (total_legend_w, legend_h), "white")
    draw   = ImageDraw.Draw(legend)

    x = pad_x
    for (color, label), w in zip(entries, text_widths):
        draw.rectangle([x, pad_y, x + swatch, pad_y + swatch],
                       fill=color, outline="black")
        draw.text((x + swatch + pad_x, pad_y),
                  label, fill="black", font=font)
        x += swatch + pad_x + w + pad_x

    # composite under the image
    out_w = max(image.width, total_legend_w)
    out_h = image.height + pad_mid + legend_h
    combined = Image.new("RGB", (out_w, out_h), "white")
    combined.paste(image, ((out_w - image.width)//2, 0))
    combined.paste(legend, ((out_w - total_legend_w)//2, image.height + pad_mid))

    return combined

In [14]:
in_dir = "./randomized_options/examples_same_second"
out_dir = "./randomized_options/examples_same_second_wlegend"

In [15]:
for fn in os.listdir(in_dir):
    img_path = os.path.join(in_dir, fn)
    
    im = add_legend_tight(Image.open(img_path), COLOR_MAPPING)
    output_path = os.path.join(out_dir, fn)

    im.save(output_path, format="PNG")